# LeetCode 2216 — Minimum Deletions to Make Array Beautiful bằng Python

Link đề: https://leetcode.com/problems/minimum-deletions-to-make-array-beautiful/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minDeletion(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi góc nhìn "xóa ít nhất" thành "**giữ lại nhiều nhất**": tìm **dãy con đẹp** dài nhất.
2. Hiểu tác dụng của việc xóa: các phần tử bên phải **dịch trái**, nên **tính chẵn lẻ của vị trí** thay đổi — phải theo dõi theo **số phần tử đã giữ**, không theo chỉ số gốc.
3. Chứng minh tham lam "**giữ phần tử nếu nó không phá cặp hiện tại**" là tối ưu bằng lập luận trao đổi.
4. Cài tham lam bằng **ngăn xếp/mảng kết quả** và bằng **bộ đếm** `O(1)` bộ nhớ; cài thêm một DP tổng quát để đối chiếu.
5. Kiểm thử đối chiếu brute force (mọi tập con) trên dữ liệu nhỏ, seed cố định.

## 2. Đề bài & ví dụ

Mảng `nums` gọi là **đẹp** nếu:
- độ dài **chẵn**;
- với mọi chỉ số **chẵn** `i`: `nums[i] != nums[i + 1]` (mỗi cặp `(0,1), (2,3), ...` gồm hai số khác nhau).

Mảng rỗng là đẹp. Bạn được **xóa** phần tử bất kỳ (các phần tử bên phải dịch sang trái). Trả về số phần tử **ít nhất** cần xóa để mảng đẹp.

**Ví dụ 1:** `[1,1,2,3,5]` → `1`: xóa một số `1` ⇒ `[1,2,3,5]` (cặp `(1,2)`, `(3,5)`).

**Ví dụ 2:** `[1,1,2,2,3,3]` → `2`: ví dụ xóa `nums[0]` và `nums[5]` ⇒ `[1,2,2,3]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import combinations, product
from typing import List

examples = [([1, 1, 2, 3, 5], 1), ([1, 1, 2, 2, 3, 3], 2)]
def is_beautiful(a):
    return len(a) % 2 == 0 and all(a[i] != a[i + 1] for i in range(0, len(a), 2))

for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp, "| đã đẹp sẵn?", is_beautiful(nums))
print(is_beautiful([1, 2, 3, 5]), is_beautiful([1, 2, 2, 3]), is_beautiful([]))

[1, 1, 2, 3, 5] -> kỳ vọng 1 | đã đẹp sẵn? False
[1, 1, 2, 2, 3, 3] -> kỳ vọng 2 | đã đẹp sẵn? False
True True True


## 3. Kiến thức nền

### 3.1 Xóa phần tử làm dịch chỉ số

Sau khi xóa `nums[k]`, phần tử `nums[k+1]` trở thành vị trí `k`. Vì vậy câu hỏi "phần tử này đang ở vị trí chẵn hay lẻ?" phụ thuộc vào **số phần tử đã giữ lại trước nó**, không phụ thuộc chỉ số gốc.

In [2]:
a = [1, 1, 2, 3, 5]
b = a[:0] + a[1:]                      # xóa a[0]
print("sau khi xóa a[0]:", b, "| a[2]=2 giờ ở vị trí", b.index(2), "(trước ở vị trí 2)")

sau khi xóa a[0]: [1, 2, 3, 5] | a[2]=2 giờ ở vị trí 1 (trước ở vị trí 2)


### 3.2 Dãy con và "giữ lại nhiều nhất"

Xóa một số phần tử = giữ lại một **dãy con** (giữ nguyên thứ tự). Số xóa ít nhất `= n − (độ dài dãy con đẹp dài nhất)`.

In [3]:
keep = [0, 2, 3, 4]                     # chỉ số giữ lại của ví dụ 1
sub = [examples[0][0][i] for i in keep]
print(sub, is_beautiful(sub), "-> xóa", len(examples[0][0]) - len(sub))

[1, 2, 3, 5] True -> xóa 1


### 3.3 Ngăn xếp/mảng kết quả để xây dần dãy con

Duyệt từ trái sang phải, `res` là dãy con đã giữ. Phần tử mới `x`:
- nếu `len(res)` **chẵn** ⇒ `x` sẽ mở một cặp mới (vị trí chẵn) ⇒ luôn giữ được;
- nếu `len(res)` **lẻ** ⇒ `x` đóng cặp với `res[−1]` ⇒ chỉ giữ nếu `x != res[−1]`.

In [4]:
res = []
for x in [1, 1, 2, 3, 5]:
    if len(res) % 2 == 0 or res[-1] != x:
        res.append(x)
print("dãy giữ lại:", res, "| độ dài lẻ -> phải bỏ phần tử cuối")

dãy giữ lại: [1, 2, 3, 5] | độ dài lẻ -> phải bỏ phần tử cuối


### 3.4 Duyệt tập con bằng `combinations` (cho brute force)

`combinations(range(n), r)` sinh mọi tập `r` chỉ số theo thứ tự tăng — đúng là mọi dãy con độ dài `r`.

In [5]:
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


### 3.5 Hai cách theo dõi vị trí: `kept` và `i − deleted`

Sau khi xử lý `i` phần tử đầu, `kept + deleted = i`. Vì vậy vị trí mới của `nums[i]` (nếu được giữ) là `kept = i − deleted`. Hai biến chứa cùng một thông tin; Cách 4 dùng `deleted`, Cách 5 dùng `kept`.

In [6]:
nums = [1, 1, 2, 3, 5]; kept = deleted = 0; last = None
for i, x in enumerate(nums):
    if kept % 2 == 0: kept += 1; last = x; act = "giữ (mở)"
    elif x != last: kept += 1; act = "giữ (đóng)"
    else: deleted += 1; act = "xóa"
    print(f"i={i} x={x}: {act:10s} kept={kept} deleted={deleted} | kept + deleted = {kept + deleted} = i + 1")

i=0 x=1: giữ (mở)   kept=1 deleted=0 | kept + deleted = 1 = i + 1
i=1 x=1: xóa        kept=1 deleted=1 | kept + deleted = 2 = i + 1
i=2 x=2: giữ (đóng) kept=2 deleted=1 | kept + deleted = 3 = i + 1
i=3 x=3: giữ (mở)   kept=3 deleted=1 | kept + deleted = 4 = i + 1
i=4 x=5: giữ (đóng) kept=4 deleted=1 | kept + deleted = 5 = i + 1


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `[1,1,2,2,3,3]`. Xây dãy đẹp từ trái sang phải, mỗi lần quyết định giữ/xóa:

| x | dãy đã giữ (trước) | độ dài | quyết định | dãy sau |
|---|---|---|---|---|
| 1 | [] | 0 (chẵn) | giữ (mở cặp) | [1] |
| 1 | [1] | 1 (lẻ) | `1 == 1` ⇒ **xóa** | [1] |
| 2 | [1] | 1 (lẻ) | `2 != 1` ⇒ giữ (đóng cặp) | [1, 2] |
| 2 | [1, 2] | 2 (chẵn) | giữ (mở cặp) | [1, 2, 2] |
| 3 | [1, 2, 2] | 3 (lẻ) | `3 != 2` ⇒ giữ | [1, 2, 2, 3] |
| 3 | [1, 2, 2, 3] | 4 (chẵn) | giữ (mở cặp) | [1, 2, 2, 3, 3] |

Cuối cùng độ dài 5 (lẻ) ⇒ xóa thêm phần tử cuối ⇒ tổng xóa `2` ✓.

**Vì sao chỉ cần nhớ phần tử mở cặp (không cần nhớ cả dãy)?** Các cặp đã đóng không bao giờ bị ảnh hưởng bởi phần tử phía sau (ta chỉ thêm vào cuối), nên quá khứ chỉ còn "để lại" đúng một thông tin cho tương lai: dãy đang chẵn, hay đang lẻ với phần tử mở là gì. Đó là toàn bộ **trạng thái** — và cũng là lý do DP ở Cách 2 chỉ cần `even` và `odd[v]`.

**Trực giác tham lam:** khi phần tử `x` đang được xét để **đóng cặp** mà bằng phần tử mở cặp, xóa `x` là lựa chọn "an toàn": phần tử mở cặp vẫn chờ, và mọi phần tử sau `x` vẫn có cơ hội đóng cặp với nó. Giữ `x` (thay vì phần tử mở cặp) cũng chẳng tốt hơn vì hai số bằng nhau.

## 5. Cách 1 — Brute Force / baseline: thử mọi dãy con

**Ý tưởng:** thử giữ lại mọi tập chỉ số, từ lớn tới nhỏ; tập đầu tiên tạo thành dãy đẹp cho đáp án `n − r`.

In [7]:
def deletions_brute(nums):
    n = len(nums)
    for r in range(n - n % 2, -1, -2):         # chỉ xét độ dài chẵn, giảm dần
        for idx in combinations(range(n), r):
            if is_beautiful([nums[i] for i in idx]):
                return n - r
    return n

for nums, exp in examples:
    got = deletions_brute(nums); print(nums, "->", got); assert got == exp

[1, 1, 2, 3, 5] -> 1
[1, 1, 2, 2, 3, 3] -> 2


**Độ phức tạp:** worst case `O(2ⁿ · n)`. Chỉ dùng cho `n ≤ ~16`.

**Nút thắt:** ta thử mọi tập trong khi có thể **quyết định từng phần tử** dựa trên trạng thái rất nhỏ (độ dài dãy đã giữ chẵn/lẻ và phần tử mở cặp).

In [8]:
rng = random.Random(0)
for n in (10, 13, 16):
    a = [rng.randint(0, 1) for _ in range(n)]
    t = time.perf_counter(); deletions_brute(a)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=10: 0.000s
n=13: 0.000s
n=16: 0.001s


## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái đủ dùng:** khi duyệt trái → phải, chỉ cần biết (i) dãy đã giữ có độ dài **chẵn** hay **lẻ**, và (ii) nếu lẻ thì **phần tử mở cặp** là gì.

**Tham lam:**
- **Nếu** độ dài đang chẵn **thì** giữ `x` (mở cặp mới).
- **Nếu** độ dài đang lẻ và `x != phần tử mở cặp` **thì** giữ `x` (đóng cặp).
- **Nếu** độ dài đang lẻ và `x == phần tử mở cặp` **thì** xóa `x`.
- Kết thúc: **nếu** độ dài lẻ **thì** xóa phần tử cuối (phần tử mở cặp dang dở).

**Bổ đề trao đổi:** tại mỗi bước, trạng thái sau tham lam **"không kém hơn"** trạng thái của bất kỳ chiến lược nào: cùng hoặc nhiều hơn số cặp đã hoàn thành. Chi tiết ở mục 10.

**Invariant (cách đếm):** `kept` = số phần tử đã giữ; `kept % 2` cho biết vị trí của phần tử tiếp theo; `last` = phần tử mở cặp khi `kept` lẻ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP tổng quát: trạng thái (chẵn) hoặc (lẻ, phần tử mở cặp)

Không tin tham lam: DP trên trạng thái. `even` = độ dài lớn nhất của dãy con đẹp (đã đóng hết cặp); `odd[v]` = độ dài lớn nhất của dãy con có độ dài lẻ, phần tử mở cặp là `v`. Với mỗi `x`:
- đóng cặp: `new_even = max(even, max_{v ≠ x} odd[v] + 1)`;
- mở cặp: `new_odd[x] = max(odd[x], even + 1)`.

Để `max_{v≠x}` nhanh, ta giữ **hai** giá trị `odd` lớn nhất (khác khóa).

In [9]:
def deletions_dp(nums):
    NEG = float("-inf")
    even = 0
    odd = {}                                         # v -> độ dài lẻ lớn nhất, mở cặp bằng v
    for x in nums:
        # max odd[v] với v != x: duyệt tối đa hai ứng viên lớn nhất
        top = sorted(odd.items(), key=lambda kv: -kv[1])[:2]
        best_other = max((val for v, val in top if v != x), default=NEG)
        new_even = max(even, best_other + 1)
        odd[x] = max(odd.get(x, NEG), even + 1)       # dùng even CŨ
        even = new_even
    return len(nums) - even

for nums, exp in examples:
    assert deletions_dp(nums) == exp
print([deletions_dp(nums) for nums, _ in examples])

[1, 2]


**Độ phức tạp Cách 2:** bản trên sắp `odd` mỗi bước ⇒ `O(n · K log K)` (`K` = số giá trị khác nhau) — chỉ để kiểm chứng. Có thể giữ top-2 tăng dần để còn `O(n)`, nhưng tham lam đơn giản hơn nhiều.

### 7.2 Cách 3 — Tham lam với mảng kết quả (ngăn xếp)

In [10]:
def deletions_stack(nums):
    res = []
    for x in nums:
        if len(res) % 2 == 0 or res[-1] != x:
            res.append(x)
    if len(res) % 2:
        res.pop()
    return len(nums) - len(res)

for nums, exp in examples:
    assert deletions_stack(nums) == exp
print([deletions_stack(nums) for nums, _ in examples])

[1, 2]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng `res`). Ưu điểm: trả về luôn **một** dãy đẹp dài nhất.

### 7.3 Cách 4 — Tham lam theo chỉ số gốc, đếm số đã xóa (bản nộp chính)

Không dựng mảng: vị trí hiện tại của `nums[i]` sau khi xóa là `i − deleted`. Nếu vị trí đó chẵn và `nums[i] == nums[i+1]` thì xóa `nums[i]` (tương đương xóa phần tử "đóng cặp" ở Cách 3 — hai số bằng nhau nên xóa cái nào cũng vậy).

In [11]:
def deletions_count(nums):
    n = len(nums); deleted = 0
    for i in range(n - 1):
        if (i - deleted) % 2 == 0 and nums[i] == nums[i + 1]:
            deleted += 1
    if (n - deleted) % 2:
        deleted += 1
    return deleted

for nums, exp in examples:
    assert deletions_count(nums) == exp
print([deletions_count(nums) for nums, _ in examples])

[1, 2]


In [12]:
class SolutionCount:
    def minDeletion(self, nums: List[int]) -> int:
        n = len(nums); deleted = 0
        for i in range(n - 1):
            if (i - deleted) % 2 == 0 and nums[i] == nums[i + 1]:
                deleted += 1
        return deleted + (n - deleted) % 2

print([SolutionCount().minDeletion(nums) for nums, _ in examples])

[1, 2]


**Độ phức tạp Cách 4:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.4 Cách 5 — Tham lam với hai biến `kept`, `last`

Giống Cách 3 nhưng chỉ nhớ **độ dài** và **phần tử mở cặp** — `O(1)` bộ nhớ và dễ chứng minh nhất.

In [13]:
def deletions_two_vars(nums):
    kept, last = 0, None
    for x in nums:
        if kept % 2 == 0:
            kept += 1; last = x                   # mở cặp
        elif x != last:
            kept += 1                             # đóng cặp
    kept -= kept % 2
    return len(nums) - kept

for nums, exp in examples:
    assert deletions_two_vars(nums) == exp
print([deletions_two_vars(nums) for nums, _ in examples])

[1, 2]


## 8. TRACE từng bước

Trace Cách 4 trên `[1, 1, 1, 2, 2, 3]` (tự dựng; kiểm bằng brute force ở dưới):

| i | nums[i], nums[i+1] | vị trí hiện tại `i − deleted` | chẵn? | bằng nhau? | deleted |
|---|---|---|---|---|---|
| 0 | 1, 1 | 0 | ✓ | ✓ ⇒ xóa | 1 |
| 1 | 1, 1 | 0 | ✓ | ✓ ⇒ xóa | 2 |
| 2 | 1, 2 | 0 | ✓ | ✗ | 2 |
| 3 | 2, 2 | 1 | ✗ | (không xét) | 2 |
| 4 | 2, 3 | 2 | ✓ | ✗ | 2 |

Còn lại `6 − 2 = 4` phần tử (chẵn) ⇒ đáp án `2`. Dãy còn lại `[1, 2, 2, 3]`.

In [14]:
def trace_count(nums):
    n = len(nums); deleted = 0
    for i in range(n - 1):
        pos = i - deleted
        act = ""
        if pos % 2 == 0 and nums[i] == nums[i + 1]:
            deleted += 1; act = "-> XÓA nums[i]"
        print(f"i={i}: ({nums[i]}, {nums[i + 1]}) vị trí hiện tại {pos} ({'chẵn' if pos % 2 == 0 else 'lẻ'}) {act} | deleted={deleted}")
    extra = (n - deleted) % 2
    print(f"còn {n - deleted} phần tử -> {'lẻ, xóa thêm 1' if extra else 'chẵn'}; đáp án {deleted + extra}")
    return deleted + extra

a = [1, 1, 1, 2, 2, 3]
print("brute force:", deletions_brute(a)); trace_count(a)

brute force: 2
i=0: (1, 1) vị trí hiện tại 0 (chẵn) -> XÓA nums[i] | deleted=1
i=1: (1, 1) vị trí hiện tại 0 (chẵn) -> XÓA nums[i] | deleted=2
i=2: (1, 2) vị trí hiện tại 0 (chẵn)  | deleted=2
i=3: (2, 2) vị trí hiện tại 1 (lẻ)  | deleted=2
i=4: (2, 3) vị trí hiện tại 2 (chẵn)  | deleted=2
còn 4 phần tử -> chẵn; đáp án 2


2

In [15]:
def trace_stack(nums):
    res = []
    for x in nums:
        if len(res) % 2 == 0:
            res.append(x); act = "mở cặp"
        elif res[-1] != x:
            res.append(x); act = "đóng cặp"
        else:
            act = "XÓA (bằng phần tử mở cặp)"
        print(f"x={x}: {act:28s} res={res}")
    if len(res) % 2:
        print(f"độ dài lẻ -> bỏ {res.pop()}")
    print("dãy đẹp giữ lại:", res, "| số xóa:", len(nums) - len(res))
    return len(nums) - len(res)

trace_stack([1, 1, 2, 2, 3, 3])

x=1: mở cặp                       res=[1]
x=1: XÓA (bằng phần tử mở cặp)    res=[1]
x=2: đóng cặp                     res=[1, 2]
x=2: mở cặp                       res=[1, 2, 2]
x=3: đóng cặp                     res=[1, 2, 2, 3]
x=3: mở cặp                       res=[1, 2, 2, 3, 3]
độ dài lẻ -> bỏ 3
dãy đẹp giữ lại: [1, 2, 2, 3] | số xóa: 2


2

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minDeletion(self, nums: List[int]) -> int:
        kept, last = 0, None                 # số phần tử đã giữ, phần tử mở cặp
        for x in nums:
            if kept % 2 == 0:                # x sẽ ở vị trí chẵn: mở cặp mới
                kept += 1
                last = x
            elif x != last:                  # x đóng cặp hợp lệ
                kept += 1
            # ngược lại: x bằng phần tử mở cặp -> xóa x
        kept -= kept % 2                     # độ dài phải chẵn
        return len(nums) - kept

for nums, exp in examples:
    assert Solution().minDeletion(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `G_i` = số phần tử tham lam giữ trong `nums[0..i−1]` và trạng thái tương ứng (chẵn, hoặc lẻ với phần tử mở `last`). Gọi `O` là một dãy con đẹp tối ưu bất kỳ, `O_i` = phần của nó trong `nums[0..i−1]`.

**Khẳng định:** với mọi `i`, số **cặp hoàn chỉnh** của tham lam `⌊G_i / 2⌋ ≥ ⌊|O_i| / 2⌋`; hơn nữa nếu hai số này bằng nhau và `O_i` đang có một phần tử mở cặp thì tham lam cũng đang có một phần tử mở cặp.

**Phác thảo:**
- Khi tham lam ở trạng thái chẵn, nó giữ ngay phần tử tiếp theo — không bao giờ "chậm" hơn trong việc mở cặp.
- Khi tham lam đang mở cặp với `last`, phần tử `x`: nếu `x != last` nó đóng cặp ngay (sớm nhất có thể); nếu `x == last` thì việc thay `last` bằng `x` không làm thay đổi gì (cùng giá trị) — nên xóa `x` không mất cơ hội nào.
- Do đó tham lam hoàn thành mỗi cặp **sớm nhất** có thể; theo quy nạp, số cặp hoàn chỉnh của nó luôn không ít hơn bất kỳ chiến lược nào.

Kết thúc: số phần tử giữ được = `2 × (số cặp hoàn chỉnh)` là lớn nhất ⇒ số xóa nhỏ nhất.

**Cách 4 tương đương Cách 5:** "vị trí hiện tại chẵn và `nums[i] == nums[i+1]`" ⇔ `nums[i]` đang mở cặp và phần tử sau bằng nó; Cách 4 xóa `nums[i]` (phần tử mở), Cách 5 xóa `nums[i+1]` (phần tử đóng) — hai số bằng nhau nên trạng thái sau như nhau.

In [17]:
# Kiểm chứng "tham lam hoàn thành cặp sớm nhất": số cặp sau mỗi tiền tố >= số cặp tối ưu của tiền tố đó
rng = random.Random(10)
for _ in range(150):
    a = [rng.randint(0, 2) for _ in range(rng.randint(1, 9))]
    kept = 0; last = None
    for i, x in enumerate(a, 1):
        if kept % 2 == 0: kept += 1; last = x
        elif x != last: kept += 1
        best_prefix = len(a[:i]) - deletions_brute(a[:i])          # dãy đẹp dài nhất của tiền tố
        assert (kept // 2) * 2 == best_prefix
print("Sau mọi tiền tố, số phần tử ghép cặp của tham lam = tối ưu của tiền tố đó (150 mảng)")

Sau mọi tiền tố, số phần tử ghép cặp của tham lam = tối ưu của tiền tố đó (150 mảng)


## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `K` = số giá trị khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi dãy con | `O(2ⁿ · n)` | `O(n)` |
| 2. DP trạng thái | `O(n · K log K)` (bản minh họa) | `O(K)` |
| 3. Tham lam + mảng | `O(n)` | `O(n)` |
| 4. Tham lam đếm xóa | `O(n)` | `O(1)` |
| 5. Hai biến | `O(n)` | `O(1)` |

In [18]:
rng = random.Random(2216)
big = [rng.randint(0, 2) for _ in range(10**5)]
for name, fn in [("stack", deletions_stack), ("count", deletions_count), ("two_vars", deletions_two_vars),
                 ("Solution", lambda a: Solution().minDeletion(a))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")

stack    : 19950 (0.005s)
count    : 19950 (0.005s)
two_vars : 19950 (0.004s)
Solution : 19950 (0.004s)


## 12. Bộ test

- Ví dụ đề.
- Biên: `n = 1` (luôn xóa 1); mọi phần tử bằng nhau (xóa hết); mảng đã đẹp (0); mảng xen kẽ độ dài lẻ (xóa 1).
- **Vét cạn** mọi mảng độ dài `1..8` trên `{0, 1, 2}` (9840 mảng) so với brute force; ngẫu nhiên lớn hơn so DP với tham lam.

In [19]:
fns = {"brute": deletions_brute, "dp": deletions_dp, "stack": deletions_stack, "count": deletions_count,
       "two_vars": deletions_two_vars, "Solution": lambda a: Solution().minDeletion(a),
       "SolutionCount": lambda a: SolutionCount().minDeletion(a)}
edge = [([7], 1), ([5, 5, 5, 5], 4), ([1, 2, 3, 4], 0), ([1, 2, 1], 1), ([1, 1], 2), ([2, 1, 1, 1, 2], 1)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums))
cnt = 0
for n in range(1, 9):
    for tup in product(range(3), repeat=n):
        a = list(tup); want = deletions_brute(a)
        for name, fn in fns.items():
            if name != "brute": assert fn(a) == want, (name, a)
        cnt += 1
print(f"Ví dụ + biên + vét cạn {cnt} mảng: OK")

Ví dụ + biên + vét cạn 9840 mảng: OK


In [20]:
rng = random.Random(42)
for _ in range(300):
    a = [rng.randint(0, rng.choice([1, 3, 10])) for _ in range(rng.randint(1, 300))]
    r = deletions_dp(a)
    assert deletions_stack(a) == deletions_count(a) == deletions_two_vars(a) == Solution().minDeletion(a) == r
print("300 test ngẫu nhiên: tham lam khớp DP tổng quát")

300 test ngẫu nhiên: tham lam khớp DP tổng quát


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Kiểm cặp theo chỉ số **gốc** (quên dịch vị trí sau khi xóa)

In [21]:
def wrong_original_parity(nums):
    n = len(nums); deleted = 0
    for i in range(n - 1):
        if i % 2 == 0 and nums[i] == nums[i + 1]:          # SAI: phải là (i - deleted) % 2
            deleted += 1
    return deleted + (n - deleted) % 2

a = next(list(t) for n in range(2, 8) for t in product(range(3), repeat=n)
         if wrong_original_parity(list(t)) != deletions_brute(list(t)))
print("phản ví dụ ngắn nhất:", a, "| sai:", wrong_original_parity(a), "| đúng:", deletions_brute(a))
print("ví dụ 2: sai cho", wrong_original_parity([1, 1, 2, 2, 3, 3]), "| đúng:", 2)

phản ví dụ ngắn nhất: [0, 0, 0] | sai: 1 | đúng: 3
ví dụ 2: sai cho 4 | đúng: 2


In [22]:
print("sửa:", Solution().minDeletion(a))

sửa: 3


### Lỗi 2 — Quên xóa phần tử cuối khi độ dài còn lại lẻ

In [23]:
def wrong_no_final(nums):
    res = []
    for x in nums:
        if len(res) % 2 == 0 or res[-1] != x: res.append(x)
    return len(nums) - len(res)                              # SAI: thiếu bước làm chẵn

print("sai:", wrong_no_final([1, 2, 3]), "| đúng:", deletions_brute([1, 2, 3]))

sai: 0 | đúng: 1


In [24]:
print("sửa:", deletions_stack([1, 2, 3]))

sửa: 1


### Lỗi 3 — Gặp hai số bằng nhau thì xóa **cả hai**

In [25]:
def wrong_delete_both(nums):
    res = []; i = 0
    while i < len(nums):
        if len(res) % 2 == 1 and res[-1] == nums[i]:
            res.pop(); i += 1                                # SAI: xóa luôn phần tử mở cặp
            continue
        res.append(nums[i]); i += 1
    if len(res) % 2: res.pop()
    return len(nums) - len(res)

a = [1, 1, 2, 3, 5]
print("sai:", wrong_delete_both(a), "| đúng:", deletions_brute(a))

sai: 3 | đúng: 1


In [26]:
print("sửa:", Solution().minDeletion(a))

sửa: 1


### Lỗi 4 — Hiểu nhầm điều kiện thành "mọi cặp kề nhau khác nhau"

Đề chỉ yêu cầu các cặp `(0,1), (2,3), ...` khác nhau; `nums[1]` và `nums[2]` được phép bằng nhau (như `[1,2,2,3]`).

In [27]:
def wrong_all_adjacent(nums):
    res = []
    for x in nums:
        if not res or res[-1] != x: res.append(x)             # SAI: cấm mọi cặp kề bằng nhau
    if len(res) % 2: res.pop()
    return len(nums) - len(res)

a = [1, 2, 2, 3]
print("sai:", wrong_all_adjacent(a), "| đúng:", deletions_brute(a))

sai: 2 | đúng: 0


In [28]:
print("sửa:", Solution().minDeletion(a))

sửa: 0


### Lỗi 5 — Đếm "số cặp bằng nhau ở vị trí chẵn" một lần trên mảng gốc rồi cộng

Mỗi lần xóa làm dịch các cặp phía sau, nên phải xử lý **tuần tự**. Đếm tĩnh trên mảng gốc cho kết quả sai.

In [29]:
def wrong_static_count(nums):
    n = len(nums)
    bad = sum(1 for i in range(0, n - 1, 2) if nums[i] == nums[i + 1])   # SAI: không tính dịch chuyển
    return bad + (n - bad) % 2

a = next(list(t) for n in range(2, 8) for t in product(range(3), repeat=n)
         if wrong_static_count(list(t)) != deletions_brute(list(t)))
print("phản ví dụ ngắn nhất:", a, "| sai:", wrong_static_count(a), "| đúng:", deletions_brute(a))

phản ví dụ ngắn nhất: [0, 0, 0] | sai: 1 | đúng: 3


In [30]:
print("sửa:", deletions_count(a))

sửa: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xóa ít nhất để thỏa điều kiện cục bộ"** ⇒ "giữ nhiều nhất", xây dãy con từ trái sang phải với **trạng thái nhỏ**.
- Khi xóa làm **dịch chỉ số**, theo dõi vị trí bằng `i − deleted` hoặc độ dài dãy đã giữ.
- Tham lam "hoàn thành cặp sớm nhất" thường tối ưu cho các bài ghép cặp liên tiếp.
- Đừng quên **ràng buộc độ dài** (chẵn) ở cuối.
- DP trạng thái nhỏ là cách tốt để kiểm tham lam.

## 15. Biến thể của bài

1. **Mọi phần tử kề nhau khác nhau** (bỏ ràng buộc cặp): xóa `run − 1` phần tử trong mỗi khối giá trị bằng nhau.
2. **Cặp phải tăng nghiêm ngặt** `nums[2i] < nums[2i+1]` (bài tập 5): tham lam "giữ phần tử mở cặp **nhỏ nhất** có thể".
3. **LeetCode 1576 / 1957** — xóa/đổi ký tự để không có ba ký tự liên tiếp giống nhau: cùng khung "xây dần".
4. **Cặp có hiệu ≥ k:** biến thể của (2), tham lam tương tự với điều kiện khác.

Minh họa biến thể 1 (tự dựng, kiểm bằng brute force):

In [31]:
def min_del_all_adjacent_distinct(nums):
    return sum(1 for i in range(1, len(nums)) if nums[i] == nums[i - 1])

def brute_all_adjacent(nums):
    n = len(nums)
    for r in range(n, -1, -1):
        for idx in combinations(range(n), r):
            sub = [nums[i] for i in idx]
            if all(sub[i] != sub[i + 1] for i in range(len(sub) - 1)):
                return n - r

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(0, 2) for _ in range(rng.randint(1, 9))]
    assert min_del_all_adjacent_distinct(a) == brute_all_adjacent(a)
print("Biến thể 'mọi cặp kề khác nhau' khớp brute force; [1,1,2,2,3,3] ->", min_del_all_adjacent_distinct([1, 1, 2, 2, 3, 3]))

Biến thể 'mọi cặp kề khác nhau' khớp brute force; [1,1,2,2,3,3] -> 3


Minh họa biến thể 4 — **cặp có hiệu ít nhất `k`** (`|nums[2i] − nums[2i+1]| ≥ k`, tự dựng). Khác bài gốc, khi phần tử mới `x` không đóng được cặp với phần tử mở `a`, **chưa chắc** nên giữ `a`: có thể `x` là phần tử mở cặp "dễ ghép" hơn. Vì vậy tham lam đơn giản có thể sai — ta dùng DP tổng quát (như Cách 2, nhưng điều kiện ghép là `|v − x| ≥ k`) và so với brute force.

In [32]:
def min_del_diff_k(nums, k):
    NEG = float("-inf"); even = 0; odd = {}
    for x in nums:
        best_other = max((val for v, val in odd.items() if abs(v - x) >= k), default=NEG)
        new_even = max(even, best_other + 1)
        odd[x] = max(odd.get(x, NEG), even + 1)
        even = new_even
    return len(nums) - even

def brute_diff_k(nums, k):
    n = len(nums)
    for r in range(n - n % 2, -1, -2):
        for idx in combinations(range(n), r):
            sub = [nums[i] for i in idx]
            if all(abs(sub[i] - sub[i + 1]) >= k for i in range(0, r, 2)):
                return n - r
    return n

def greedy_diff_k(nums, k):
    kept, last = 0, None
    for x in nums:
        if kept % 2 == 0: kept += 1; last = x
        elif abs(x - last) >= k: kept += 1
    return len(nums) - (kept - kept % 2)

rng = random.Random(6)
bad = None
for _ in range(400):
    a = [rng.randint(0, 6) for _ in range(rng.randint(1, 9))]; k = rng.randint(1, 4)
    assert min_del_diff_k(a, k) == brute_diff_k(a, k)
    if bad is None and greedy_diff_k(a, k) != brute_diff_k(a, k): bad = (a, k)
print("DP cho 'hiệu >= k' khớp brute force (400 bộ)")
print("tham lam kiểu bài gốc SAI ở:", bad, "->", greedy_diff_k(*bad), "vs", brute_diff_k(*bad))

DP cho 'hiệu >= k' khớp brute force (400 bộ)
tham lam kiểu bài gốc SAI ở: ([0, 1, 3, 4, 5, 3, 0, 3], 2) -> 4 vs 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy con | brute force | `O(2ⁿ·n)` | `O(n)` | kiểm tra |
| 2. DP trạng thái | chẵn / (lẻ, phần tử mở) | `O(n·K)` | `O(K)` | kiểm tham lam, biến thể khó |
| 3. Tham lam + mảng | xây dãy đẹp | `O(n)` | `O(n)` | cần trả dãy kết quả |
| 4. Đếm xóa | vị trí `i − deleted` | `O(n)` | `O(1)` | ngắn gọn |
| 5. Hai biến | `kept`, `last` | `O(n)` | `O(1)` | **nộp bài**, dễ chứng minh |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `minDeletion([4, 4, 4])` và `minDeletion([1, 2, 1, 2])`.
2. **Tính tay:** trace Cách 5 trên `[3, 3, 3, 4, 4, 5, 5]`.
3. **Điền TODO:** `beautiful_result(nums)` trả về **một** dãy đẹp dài nhất (không chỉ số xóa).
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `min_del_increasing_pairs(nums)` — mảng "đẹp" là độ dài chẵn và `nums[2i] < nums[2i+1]`. Tìm số xóa ít nhất. Kiểm bằng brute force.

In [33]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def beautiful_result(nums):
    # TODO
    pass

In [34]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_min_deletion(nums):
    # TODO
    pass

In [35]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def min_del_increasing_pairs(nums):
    # TODO: khi đang mở cặp với a, gặp x: nếu x > a thì đóng cặp; ngược lại thay a bằng x (nhỏ hơn hoặc bằng) và tính 1 lần xóa
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `[4,4,4]`: chỉ giữ được 1 số 4 — độ dài lẻ ⇒ giữ 0 ⇒ xóa 3. (2) `[1,2,1,2]`: đã đẹp ⇒ 0. (3) Kiểm bằng `is_beautiful`.

**Bài 2.** (1) 3 mở; 3, 3 bị xóa; 4 đóng. (2) 4 mở; 5 đóng. (3) 5 mở, độ dài lẻ ⇒ bỏ ⇒ giữ 4, xóa 3.

**Bài 3.** (1) Dùng Cách 3. (2) Trả `res`. (3) Kiểm `is_beautiful(res)` và độ dài.

**Bài 4.** (1) `deleted = 0`. (2) Điều kiện `(i − deleted) % 2 == 0 and nums[i] == nums[i+1]`. (3) Cộng `(n − deleted) % 2`.

**Bài 5.** (1) Phần tử mở cặp càng **nhỏ** càng dễ đóng cặp. (2) Gặp `x ≤ a` thì thay `a` bằng `x` (xóa một phần tử). (3) Kiểm bằng brute force mọi dãy con.

## 19. Đáp án tham khảo

In [36]:
# Bài 1, 2
assert Solution().minDeletion([4, 4, 4]) == 3 and Solution().minDeletion([1, 2, 1, 2]) == 0
assert Solution().minDeletion([3, 3, 3, 4, 4, 5, 5]) == 3 == deletions_brute([3, 3, 3, 4, 4, 5, 5])
print("Bài 1: 3 và 0 | Bài 2: 3")
trace_stack([3, 3, 3, 4, 4, 5, 5])

Bài 1: 3 và 0 | Bài 2: 3
x=3: mở cặp                       res=[3]
x=3: XÓA (bằng phần tử mở cặp)    res=[3]
x=3: XÓA (bằng phần tử mở cặp)    res=[3]
x=4: đóng cặp                     res=[3, 4]
x=4: mở cặp                       res=[3, 4, 4]
x=5: đóng cặp                     res=[3, 4, 4, 5]
x=5: mở cặp                       res=[3, 4, 4, 5, 5]
độ dài lẻ -> bỏ 5
dãy đẹp giữ lại: [3, 4, 4, 5] | số xóa: 3


3

In [37]:
# Bài 3, 4
def beautiful_result_ans(nums):
    res = []
    for x in nums:
        if len(res) % 2 == 0 or res[-1] != x: res.append(x)
    if len(res) % 2: res.pop()
    return res

my_min_deletion_ans = deletions_count
rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(0, 2) for _ in range(rng.randint(1, 10))]
    r = beautiful_result_ans(a)
    assert is_beautiful(r) and len(a) - len(r) == deletions_brute(a) == my_min_deletion_ans(a)
print("Bài 3, 4: OK; ví dụ 2 ->", beautiful_result_ans([1, 1, 2, 2, 3, 3]))

Bài 3, 4: OK; ví dụ 2 -> [1, 2, 2, 3]


In [38]:
# Bài 5
def min_del_increasing_pairs_ans(nums):
    kept, a = 0, None
    for x in nums:
        if kept % 2 == 0:
            kept += 1; a = x                       # mở cặp
        elif x > a:
            kept += 1                              # đóng cặp
        else:
            a = x                                  # thay phần tử mở bằng số nhỏ hơn (xóa phần tử mở cũ)
    kept -= kept % 2
    return len(nums) - kept

def brute_increasing_pairs(nums):
    n = len(nums)
    for r in range(n - n % 2, -1, -2):
        for idx in combinations(range(n), r):
            sub = [nums[i] for i in idx]
            if all(sub[i] < sub[i + 1] for i in range(0, r, 2)):
                return n - r
    return n

rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(0, 4) for _ in range(rng.randint(1, 10))]
    assert min_del_increasing_pairs_ans(a) == brute_increasing_pairs(a)
print("Bài 5: OK; [5, 3, 4, 1, 2] ->", min_del_increasing_pairs_ans([5, 3, 4, 1, 2]))

Bài 5: OK; [5, 3, 4, 1, 2] -> 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "xây dãy con ghép cặp, tham lam":**

```python
kept, last = 0, None
for x in nums:
    if kept % 2 == 0: kept += 1; last = x      # mở cặp
    elif ok(last, x): kept += 1                # đóng cặp
    # else: xóa x (hoặc cập nhật last nếu có lợi)
kept -= kept % 2
return len(nums) - kept
```

In [39]:
def template(nums, ok=lambda a, b: a != b):
    kept, last = 0, None
    for x in nums:
        if kept % 2 == 0: kept += 1; last = x
        elif ok(last, x): kept += 1
    kept -= kept % 2
    return len(nums) - kept
print([template(a) for a, _ in examples])

[1, 2]


**Checklist:**

- [ ] Theo dõi vị trí **sau khi xóa** (`kept % 2` hoặc `i − deleted`), không theo chỉ số gốc.
- [ ] Chỉ các cặp `(2i, 2i+1)` phải khác nhau (không phải mọi cặp kề).
- [ ] Gặp phần tử bằng phần tử mở cặp ⇒ xóa **một** phần tử.
- [ ] Cuối cùng làm độ dài **chẵn** (xóa thêm 1 nếu lẻ).
- [ ] Đã thử: `n = 1`, mọi phần tử bằng nhau, mảng đã đẹp.